In [12]:
import os

# Clone the GitHub repository to access the data file
repo_url = 'https://github.com/d1vyesh-27/flyrank-ai-internship-ml'
repo_name = repo_url.split('/')[-1]

if not os.path.exists(repo_name):
    !git clone {repo_url}
else:
    print(f"Repository '{repo_name}' already cloned.")

# Change the current working directory to the cloned repository
os.chdir(repo_name)

Cloning into 'flyrank-ai-internship-ml'...
remote: Enumerating objects: 113, done.
remote: Counting objects: 100% (113/113), done.
remote: Compressing objects: 100% (84/84), done.
remote: Total 113 (delta 29), reused 78 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (113/113), 1.85 MiB | 9.40 MiB/s, done.
Resolving deltas: 100% (29/29), done.


# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two signal checks + my rule

Before encoding anything, I check that the assumptions behind my rule are real. Both signals below are tied to real FlyRank flag logic:

- **Signal #1 — CTR vs position** (behind the CTR-fix flag logic: position tier strongly determines expected CTR).
- **Signal #2 — Volume / quick-win** (behind the quick-win flag logic: high-impression pages with low CTR are worth fixing).

Each gets a one-word verdict: **CONFIRMED / OPPOSITE / MIXED / FALSE**.

In [13]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} cols")
print(f"Clients: {df['client_id'].nunique()}")
print("One row = one content item (page), trailing 90-day metrics.")

Shape: 30,000 rows x 44 cols
Clients: 32
One row = one content item (page), trailing 90-day metrics.


### 1a. Distributions first

*Traffic metrics are heavy-tailed — that's why tests use weighted rates and buckets, not raw correlations.*

In [14]:
print(df["ctr"].describe().round(2))
print()
print(df["impressions_90d"].describe().round(2))
print()
print("avg_position == 0 (no position data):", (df["avg_position"] == 0).sum())
print(df["position_tier"].value_counts())
print(df["impression_tier"].value_counts())

count    30000.00
mean         0.51
std          3.28
min          0.00
25%          0.00
50%          0.07
75%          0.29
max        100.00
Name: ctr, dtype: float64

count     30000.00
mean       5200.37
std       16838.02
min           1.00
25%          81.00
50%         731.00
75%        3615.25
max      517715.00
Name: impressions_90d, dtype: float64

avg_position == 0 (no position data): 1205
position_tier
page_1      11814
striking     7304
page_3_5     7242
top_3        2321
deep         1319
Name: count, dtype: int64
impression_tier
low          11248
moderate     10469
good          7205
excellent     1078
Name: count, dtype: int64


### 1b. Signal check #1 — CTR vs position (behind the CTR-fix flag)

**Claim:** better position → higher CTR. **Test:** weighted CTR per `position_tier` (total clicks ÷ total impressions; drop `avg_position == 0` rows).

**Verdict: CONFIRMED.** 0.49% (`top_3`) → 0.04% (`deep`), a 12x range. Position drives CTR, so compare pages only within tier. *Caveat: `page_1` and `striking` tie at 0.35% — doesn't change the verdict.*

In [15]:
t1 = df[df["avg_position"] > 0]

test1 = (
    t1.groupby("position_tier", observed=True)
    .apply(
        lambda g: pd.Series(
            {
                "n": len(g),
                "weighted_ctr_pct": g["clicks_90d"].sum() / g["impressions_90d"].sum() * 100,
            }
        ),
        include_groups=False,
    )
    .round(3)
)
print(test1.to_string())
print("\nVERDICT: CONFIRMED — position tracks CTR; compare pages within tier.")

                     n  weighted_ctr_pct
position_tier                           
deep            1319.0             0.041
page_1         11814.0             0.350
page_3_5        7242.0             0.155
striking        7304.0             0.347
top_3           1116.0             0.489

VERDICT: CONFIRMED — position tracks CTR; compare pages within tier.


### 1c. Signal check #2 — Volume / quick-win (behind the quick-win flag)

**Claim:** high-impression pages with weak CTR are common enough to rank. **Test:** per `impression_tier`, median CTR + share below 1% CTR.

**Verdict: CONFIRMED.** 8,283 pages at 3,000+ impressions, ~96% under 1% CTR — a large pool to rank. *Caveat: low CTR is nearly universal, so the rule must measure it against the page's own position tier.*

In [16]:
test2 = (
    df.groupby("impression_tier", observed=True)
    .apply(
        lambda g: pd.Series(
            {
                "n": len(g),
                "median_ctr_pct": g["ctr"].median(),
                "share_ctr_lt_1pct": round((g["ctr"] < 1).mean() * 100, 2),
            }
        ),
        include_groups=False,
    )
)
print(test2.to_string())
print("\nVERDICT: CONFIRMED — a large high-volume / low-CTR pool exists to rank.")

                       n  median_ctr_pct  share_ctr_lt_1pct
impression_tier                                            
excellent         1078.0            0.22              96.29
good              7205.0            0.21              95.91
low              11248.0            0.00              90.36
moderate         10469.0            0.12              97.23

VERDICT: CONFIRMED — a large high-volume / low-CTR pool exists to rank.


## 1d. My rule and its reason codes

**The rule, in plain words:** *A page is worth reviewing if it gets enough visibility (impressions) but its click rate sits well below what other pages in the same position tier achieve.*

- **Score** = (tier-expected CTR − page CTR) × log(impressions) — higher = review first. The log keeps heavy-tailed volume from dominating.
- **Reason code** = `ctr_below_tier_expected` (one code per row, matching the assignment).
- **Action label** = `review_ctr` (rewrite title/meta, improve intent match, or monitor).

The queue in section 2 encodes exactly this.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [17]:
import numpy as np
from pathlib import Path

tier_expected = (
    t1.groupby("position_tier", observed=True)
    .apply(
        lambda g: g["clicks_90d"].sum() / g["impressions_90d"].sum() * 100,
        include_groups=False,
    )
    .rename("tier_expected_ctr")
)

queue = t1.copy()
queue["tier_expected_ctr"] = queue["position_tier"].map(tier_expected)
queue["score"] = (queue["tier_expected_ctr"] - queue["ctr"]) * np.log1p(queue["impressions_90d"])
queue["reason_code"] = "ctr_below_tier_expected"
queue["action_label"] = "review_ctr"
queue = queue.sort_values("score", ascending=False)

out_cols = ["content_id", "client_id", "score", "reason_code", "action_label", "position_tier", "tier_expected_ctr", "ctr", "impressions_90d"]
out_path = Path("work/outputs/baseline_action_score.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)
queue[out_cols].to_csv(out_path, index=False)
print(f"Wrote {len(queue):,} ranked pages to {out_path}")

Wrote 28,795 ranked pages to work/outputs/baseline_action_score.csv


## 3. Top-10 review

*For each of the top 10: action, reason code, confidence note, and what would make it wrong.*

*All ten are `top_3` pages with CTR 0.01–0.15% vs a tier expected of 0.49% — the rule is ranking the intended pattern. For each: action, why it's there, what would make it wrong.*

| # | Action | Why it's there | What would make it wrong |
|---|---|---|---|
| 1 | review_ctr | 272k impressions at pos 2.3, CTR 0.03% (~16x under tier) | Impressions from one low-intent query; or newly ranking with clicks lagging (trend up) |
| 2 | review_ctr | 128k impressions, CTR 0.01% | Trend +5,426% — page just surged into the position; impressions may outrun clicks for now |
| 3 | review_ctr | 150k impressions at pos 2.9, CTR 0.07%, stable | Single broad query inflating impressions without click intent |
| 4 | review_ctr | 15k impressions, CTR 0.02%, transactional, only 4 sessions | Query mismatch: transactional searchers not finding what they want; 4 sessions = weak evidence |
| 5 | review_ctr | 26k impressions at pos 0.7 (top of page 1), CTR 0.05% | Hard to argue against — near #1 and near-zero CTR; unlikely but maybe a brand/zero-result SERP |
| 6 | review_ctr | 509k impressions, CTR 0.15%, 785 clicks, 571 sessions | Volume is real; but could be feature-rich SERP (images/PAAs) suppressing clicks site-wide |
| 7 | review_ctr | 53k impressions, CTR 0.08%, but word_count not measured | NaN word_count; no content-depth context to confirm the fix |
| 8 | review_ctr | 12k impressions, CTR 0.02%, commercial, 3 clicks | Commercial intent with 3 clicks; could be a SERP that rarely shows this URL |
| 9 | review_ctr | 12k impressions, CTR 0.02%, trend −97.7% | Impressions collapsing — the opportunity may self-resolve before review |
| 10 | review_ctr | 25k impressions at pos 1.6, CTR 0.06% | Trend −73%: shrinking impression base; may not stay a candidate |

*Confidence note: highest-volume rows (1, 3, 6) are the safest picks; rows 4, 8, 9 carry low-session / collapsing-trend caveats.*

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*
## 4. Weak picks + leakage check

*Three of the top ten are the weakest: rows 9, 5, 8 — high impression base but collapsing trend and/or thin evidence. In each case the "what would make it wrong" caveat is the more likely story.*

| Rank | content_id | clicks | sessions | trend_pct | word_count | Why weak |
|---|---|---|---|---|---|---|
| 9 | content_998f6f88784c | 3 | 8 | −97.7% | 2,964 | Impressions collapsing to near zero — opportunity may vanish before an editor acts |
| 5 | content_d225ec9f3d46 | 14 | 10 | −86.8% | 2,618 | Shrinking base + only 10 sessions; the low CTR may be a fading-SERP artifact |
| 8 | content_b7bd590fe572 | 3 | 8 | −86.5% | NaN | Commercial page, 3 clicks, word_count unmeasured — no depth signal to confirm the fix |
| 7 | content_8053a66bd6ac | 40 | 52 | −89.8% | NaN | Strong volume but word_count unmeasured; declining fast |
| 10 | content_f4e210ee0c27 | 16 | 11 | −73.4% | 4,395 | Low sessions; trend down — may not persist as a candidate |

**Leakage check:**
- The score uses only `impressions_90d`, `ctr`, `avg_position`/`position_tier` — all observable and known at the decision moment.
- No label-derived inputs: `trend_direction` / `trend_pct` / `is_declining_label` are **not** features (they're used above only to flag weak picks, after the fact).
- No future windows: the CSV is built from a single 90-day trailing window; nothing from a later period feeds the score.
- No product flags: `health_score`-style columns are not in the dataset; the reason code is our own `ctr_below_tier_expected`.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.